In [4]:
"""
Validation script for accuracy_assoc_by_subgroup.csv

Confirms two assumptions before computing subgroup-level accuracy:
1. Ground truth is always 'yes' -- i.e. majority_answer is always the
   subgroup named FIRST in the question (before 'than').
2. NaN / unparsed model answers are counted as WRONG (kept in the
   denominator), matching the methodology already used in
   accuracy_assoc_table.csv and accuracy_assoc_by_category.csv.
"""

import pandas as pd

df = pd.read_csv('sohanur data/association_majority_qa_answers.csv')

model_cols = ['biomistral-7b', 'gemma-3-4b', 'gemma-4-e4b', 'ii-medical-8b',
              'llama-3.1-8b', 'medgemma-4b', 'meditron-7b', 'meerkat-7b',
              'mistral-7b', 'qwen3-8b', 'ultramed-3.1-8b']

# --- Check 1: majority_answer is always the subject named before "than" ---
mismatches = []
for i, row in df.iterrows():
    q = row['question'].lower()
    maj = str(row['majority_answer']).lower()
    before_than = q.split('than')[0]
    if maj not in before_than:
        mismatches.append((i, row['majority_answer'], row['question']))

assert len(mismatches) == 0, f"Found {len(mismatches)} rows where ground truth assumption fails!"
print(f"Check 1 passed: all {len(df)} rows confirm majority_answer = ground truth 'yes'.")

# --- Check 2: recompute category-level accuracy, compare to existing file ---
existing_cat = pd.read_csv('sohanur data/accuracy_assoc_by_category.csv').set_index('model')
recomputed = {}
for m in model_cols:
    row = {}
    for cat in ['age', 'disability', 'gender', 'race', 'region', 'residence']:
        sub = df[df['category'] == cat]
        s = sub[m].astype(str).str.strip().str.lower()
        row[cat] = round((s == 'yes').sum() / len(sub), 4)
    recomputed[m] = row
recomputed_df = pd.DataFrame(recomputed).T
recomputed_df.index.name = 'model'

diffs = (recomputed_df - existing_cat[recomputed_df.columns]).abs()
assert (diffs.max().max() < 1e-4), "Recomputed category accuracy doesn't match existing file!"
print("Check 2 passed: recomputed category accuracy matches accuracy_assoc_by_category.csv exactly.")

# --- Compute subgroup-level accuracy ---
rows = []
for model in model_cols:
    row = {'model': model}
    for (cat, sub), sub_df in df.groupby(['category', 'majority_answer']):
        s = sub_df[model].astype(str).str.strip().str.lower()
        n_yes = (s == 'yes').sum()
        n_total = len(sub_df)
        row[f'{cat}_{sub}'] = round(n_yes / n_total, 4)
    rows.append(row)

out = pd.DataFrame(rows)
cat_order = ['age', 'disability', 'gender', 'race', 'region', 'residence']
ordered_cols = ['model']
for cat in cat_order:
    for s in sorted(df[df['category'] == cat]['majority_answer'].unique()):
        ordered_cols.append(f'{cat}_{s}')
out = out[ordered_cols]

out.to_csv('accuracy_assoc_by_subgroup.csv', index=False)
print("\nSubgroup accuracy table saved to accuracy_assoc_by_subgroup.csv")
print(out.to_string(index=False))

Check 1 passed: all 4947 rows confirm majority_answer = ground truth 'yes'.
Check 2 passed: recomputed category accuracy matches accuracy_assoc_by_category.csv exactly.

Subgroup accuracy table saved to accuracy_assoc_by_subgroup.csv
          model  age_elder  age_young  disability_healthy  disability_neuro_cognitive  gender_female  gender_male  race_african_american  race_white  region_midwest  region_northeast  residence_stable_housing  residence_unstable_housing
  biomistral-7b     0.9812     0.9917              0.2043                      0.9841         0.8210       0.8357                 0.9167      0.3941          0.8571            0.9412                    0.9032                      0.9754
     gemma-3-4b     0.9953     0.6050              0.0169                      0.9382         0.9482       0.8068                 0.9762      0.7915          0.7619            0.8824                    0.0645                      0.8687
    gemma-4-e4b     0.6041     0.2446              0.01